## 2.4 提示词基线实验

对应 PDF **Problem (prompting_baselines)**：用三种提示词把 base 模型在 GSM8K 上跑一遍，
统计三类输出，再看模型在各提示词下表现成什么样。

这一节不训练，整条链是直的：

**读数据 → 套模板 → 发请求 → 打分 → 统计**

得到的分数就是第 3 节 GRPO 要对比的基线。

In [ ]:
import os
import sys

# 仓库根目录，后面所有相对路径都相对它
os.chdir(r"C:\Users\intangible\Desktop\easy-llm-alignment")

# 公共模块（drgrpo_grader、vllm_utils）在根目录下，加进搜索路径才 import 得到
sys.path.insert(0, os.getcwd())

print("工作目录:", os.getcwd())
# 工作目录: C:\Users\intangible\Desktop\easy-llm-alignment


### 2.4.1 读数据集

GSM8K 每条样本只有两个字段：`question` 和 `answer`。
`answer` 就是模型该学会写的东西——一段推理过程，末尾跟 `####` 和最终答案。

打分只认 `####` 后面那个数，所以读进来顺手切掉，后面不用再处理。

In [ ]:
import json
from pathlib import Path


def load_gsm8k(path):
    '''
    读 GSM8K 的 jsonl，返回 [{"question": str, "gold": str}]

    path: str | Path  train.jsonl 或 test.jsonl
    '''
    examples = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if not line.strip():                       # 跳过空行
                continue
            record = json.loads(line)
            examples.append({
                "question": record["question"],
                # answer 形如 "{推理过程} #### 18"，rsplit 从右边切一次，取最后那段
                "gold": record["answer"].rsplit("####", 1)[-1].strip(),
            })
    return examples

In [ ]:
examples = load_gsm8k("data/gsm8k/test.jsonl")

print("测试集条数:", len(examples))
print("题目:", examples[0]["question"][:70], "...")
print("标准答案:", repr(examples[0]["gold"]))

# 测试集条数: 1319
# 题目: Natalia sold clips to 48 of her friends in April, and then she sold half as many cl ...
# 标准答案: '18'

### 2.4.2 组装提示词

三份模板都是纯文本，唯一的占位符是 `{question}`，用 `str.format` 填进去即可。

注意不能用 `str.replace`：`r1_zero.prompt` 的正文里有 "The User asks a question" 这句话，
`replace` 会把里面的 `question` 也一并换掉。

In [ ]:
def build_prompt(template_name, question):
    '''
    读模板并把 {question} 换成题目

    template_name: str  prompts/ 下的文件名
    question: str       题目原文
    '''
    template = Path("prompts", template_name).read_text(encoding="utf-8")
    return template.format(question=question)

In [ ]:
for name in ["question_only.prompt", "r1_zero.prompt", "r1_zero_three_shot_gsm8k.prompt"]:
    prompt = build_prompt(name, "Natalia sold clips to 48 of her friends in April...")
    print(f"{name:34} {len(prompt):>5} 字符")

# question_only.prompt                 98 字符   ← 一句问题 + 一句格式要求
# r1_zero.prompt                      490 字符   ← 加了 CoT 指令和 <think> 开标签
# r1_zero_three_shot_gsm8k.prompt    1416 字符   ← 再塞三对完整问答

### 2.4.3 起服务

`VLLMServer` 只管两件事：起进程、发请求。`launch_server` 决定它要不要连进程一起拉。

| | 谁起服务 | 适用范围 |
|---|---|---|
| `launch_server=True` | 脚本自己拉起 `vllm serve` | Linux / WSL |
| `launch_server=False` | 服务器上手动起，本地连过去 | 客户 / 服务器分离（Windows 走这条） |

第二种的完整走法见 2.4 节正文。

In [ ]:
from vllm_utils import VLLMServer

MODEL_ID = "allenai/OLMo-2-0425-1B"

# True 只在本机（Linux）用；Windows 上填 False，连服务器
LAUNCH_SERVER = False
HOST = "127.0.0.1"
PORT = 8001          # 走 SSH 隧道填本地端口；服务器那边监听 8000
GPU = 0

server = VLLMServer(
    model_id=MODEL_ID,
    host=HOST,
    port=PORT,
    gpu=GPU,
    launch_server=LAUNCH_SERVER,
)
# server.start()     # 这一步会等模型加载完，1B 模型通常一两分钟

### 2.4.4 采样参数

文档 3.4 给的三组值：temperature 1.0、top-p 1.0、最长 512 个 token。
top-p 用 vLLM 的默认值 1.0，所以不用显式传。

两处容易踩：

- `"n"` 是必填键，`generate_completions` 内部一定会读它，只采一条也得写 1；
- `stop` 只给 r1_zero 系列。question_only 没要求输出 `</answer>`，给它设这个停止符会把正常回答截断。

In [ ]:
MAX_TOKENS = 512
TEMPERATURE = 1.0
SEED = 0


def sampling_params(mode):
    '''
    构造传给 generate_completions 的采样参数

    mode: str  question_only / r1_zero / r1_zero_three_shot
    '''
    params = {"max_tokens": MAX_TOKENS, "temperature": TEMPERATURE, "seed": SEED, "n": 1}
    if mode != "question_only":
        params["stop"] = ["</answer>"]
        params["include_stop_str_in_output"] = True
    return params

In [ ]:
for mode in ["question_only", "r1_zero"]:
    print(mode, "->", sampling_params(mode))

# question_only -> {'max_tokens': 512, 'temperature': 1.0, 'seed': 0, 'n': 1}
# r1_zero       -> {'max_tokens': 512, 'temperature': 1.0, 'seed': 0, 'n': 1,
#                   'stop': ['</answer>'], 'include_stop_str_in_output': True}

### 2.4.5 打分

模型答对的方式有很多种：`<answer> 1/2 </answer>`、`<answer> 0.5 </answer>`、
甚至一句 `The answer is 0.5`。字符串比对没法覆盖，得先解析再判等，这就是 reward fn 干的事。

两种提示词的格式期望不同，必须按模式选对应的函数：r1_zero 系列看 `<think>` / `<answer>` 标签，
question_only 看 `\boxed{}`。

In [ ]:
from drgrpo_grader import question_only_reward_fn, r1_zero_reward_fn


def score(mode, response, gold):
    '''
    给单条输出打分，返回 {format_reward, answer_reward, reward}

    mode: str      决定用哪个 reward fn
    response: str  模型生成的文本
    gold: str      标准答案（只要答案本身，不带推理过程）
    '''
    if mode == "question_only":
        return question_only_reward_fn(response, gold)
    return r1_zero_reward_fn(response, gold)

In [ ]:
# 三种典型的模型输出
good = "<think> 48/2=24，48+24=72 </think> <answer> 72 </answer>"   # 格式对、答案对
wrong = "<think> 48+24=62 </think> <answer> 62 </answer>"          # 格式对、答案错
naked = "The answer is 72."                                        # 没标签，格式就不对

for name, resp in [("格式对·答案对", good), ("格式对·答案错", wrong), ("没有标签", naked)]:
    print(f"{name:10}", score("r1_zero", resp, "72"))

# 格式对·答案对 {'format_reward': 1.0, 'answer_reward': 1.0, 'reward': 1.0}
# 格式对·答案错 {'format_reward': 1.0, 'answer_reward': 0.0, 'reward': 0.0}
# 没有标签     {'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0}

注意第三行：模型其实答对了（The answer is 72），但没按格式写，解析器抓不到答案，
于是 format 和 answer 都是 0。这类样本正是作业 (a) 要人工翻看的那一批——
**看起来分数很低，实际有多少是真答错的，得自己数。**

文档 3.3 规定了不给 partial credit，所以 `reward` 永远等于 `answer_reward`。

### 2.4.6 跑一遍

一套提示词跑完整个测试集，就是：批量套模板 → 一次发出去 → 逐条打分 → 拼成行。

In [ ]:
MODES = {
    "question_only": "question_only.prompt",
    "r1_zero": "r1_zero.prompt",
    "r1_zero_three_shot": "r1_zero_three_shot_gsm8k.prompt",
}

BATCH_SIZE = 64      # 每次 HTTP 请求塞多少条，只决定分块，不影响返回顺序


def run_mode(server, mode, examples):
    '''
    一套提示词跑完整个测试集，返回逐条结果

    server: VLLMServer  已经 start 过的服务
    mode: str           提示词类型
    examples: list       load_gsm8k 的返回值
    '''
    prompts = [build_prompt(MODES[mode], ex["question"]) for ex in examples]
    completions = server.generate_completions(prompts, sampling_params(mode), BATCH_SIZE)
    assert len(completions) == len(prompts)      # 返回顺序与 prompts 一一对应

    rows = []
    for example, prompt, completion in zip(examples, prompts, completions):
        rows.append({
            "mode": mode,
            "question": example["question"],
            "gold": example["gold"],
            "prompt": prompt,
            "response": completion.text,
            "finish_reason": completion.finish_reason,
            "num_tokens": len(completion.token_ids),
            **score(mode, completion.text, example["gold"]),
        })
    return rows

In [ ]:
# 正式跑法（需要服务已经起来，全量 1319 条会跑一阵）
# rows = run_mode(server, "r1_zero", examples[:5])
# rows[0]["response"][-200:]

# 调试时先跑 5 条看看链路通不通：
# rows = run_mode(server, "r1_zero", examples[:5])
# print(rows[0]["gold"], rows[0]["answer_reward"])

### 2.4.7 统计

作业 (a) 要的三分类，就是把 `(format_reward, answer_reward)` 的四种组合数一遍。
实际上只会有三种：`(1,1)`、`(1,0)`、`(0,0)`——格式不对时解析器拿不到答案，answer 必然是 0。

准确率按文档 3.3 的口径：total reward 就是 answer reward，所以答对条数除以总条数。

In [ ]:
def summarize(rows):
    '''
    三分类计数 + 准确率 + 平均生成长度

    rows: list  run_mode 的返回值
    '''
    num_fmt1_ans1 = sum(r["format_reward"] == 1 and r["answer_reward"] == 1 for r in rows)
    num_fmt1_ans0 = sum(r["format_reward"] == 1 and r["answer_reward"] == 0 for r in rows)
    num_fmt0_ans0 = sum(r["format_reward"] == 0 and r["answer_reward"] == 0 for r in rows)
    return {
        "num_fmt1_ans1": num_fmt1_ans1,
        "num_fmt1_ans0": num_fmt1_ans0,
        "num_fmt0_ans0": num_fmt0_ans0,
        "accuracy": num_fmt1_ans1 / len(rows) if rows else 0.0,
        "average_num_tokens": sum(r["num_tokens"] for r in rows) / len(rows) if rows else 0.0,
        "num_truncated": sum(r["finish_reason"] == "length" for r in rows),
    }

In [ ]:
# 造三行假数据，验证统计口径
fake = [
    {"format_reward": 1.0, "answer_reward": 1.0, "num_tokens": 120, "finish_reason": "stop"},
    {"format_reward": 1.0, "answer_reward": 0.0, "num_tokens": 512, "finish_reason": "length"},
    {"format_reward": 0.0, "answer_reward": 0.0, "num_tokens": 40,  "finish_reason": "stop"},
]
print(summarize(fake))

# {'num_fmt1_ans1': 1, 'num_fmt1_ans0': 1, 'num_fmt0_ans0': 1,
#  'accuracy': 0.3333..., 'average_num_tokens': 224.0, 'num_truncated': 1}

三分类加起来要等于总条数，这是个好用的自检：
`num_fmt1_ans1 + num_fmt1_ans0 + num_fmt0_ans0 == len(rows)`。

如果对不上，说明出现了第四种组合 `(format=0, answer=1)`，那意味着解析器判错但答案对——
这种样本要单独挑出来看。

---

完整可运行的脚本在 `scripts/prompting_baselines.py`，包含落盘、分批、两类样本单独导出。
这个 notebook 只把关键函数拆开讲，跑正式结果用它。